---

# C4. Exercițiu individual: construirea unui mini-prompt de adnotare

În acest exercițiu construiești un prompt mic de adnotare pentru comentarii politice.
- Intelegem cum se construiește un prompt: rol, variabile, definiții, reguli și format JSON.
- Alegemdouă axe proprii sau două axe din curs și vei testa promptul pe 5 comentarii.


## Pasul 0 . Configurare

In [1]:
import os, json, re, random
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
# caută .env urcând din folderul curent

ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")

# DeepSeek
deepseek_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)
DEEPSEEK_MODEL = "deepseek-chat"
# Gemini prin OpenAI-compatible API
gemini_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

GEMINI_MODEL = "gemini-2.5-flash-lite"
# alegem modelul pentru demo

USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Root project:", ROOT)
print("DeepSeek key:", os.getenv("DEEPSEEK_API_KEY") is not None)
print("Gemini key:", os.getenv("GEMINI_API_KEY") is not None)
print("Model folosit:", model_now)
print("OK")

Root project: c:\Users\Ionela\Downloads\MCS\Proiect Inginerie AI\echochamber-project-team-5
DeepSeek key: True
Gemini key: True
Model folosit: gemini-2.5-flash-lite
OK


## Corpus

In [2]:
import pandas as pd
import random

corpus = pd.read_json("../../data/cleaned/corpus_youtube_sample.jsonl", lines=True)

print(len(corpus), "comentarii")
print("Câmpuri:", list(corpus.columns))

for _, c in corpus.sample(3).iterrows():
    print(f"[{c['source_channel'][:30]}] {c['text'][:80]}")

420 comentarii
Câmpuri: ['id', 'source_channel', 'video_title', 'text']
[europafmromania] Catu sa plateasca, daca nu se pruc probe ale unei decizii la nivel UE. Si nici d
[TuDecizi-s3g] Ce tot dati atâta importanță cum ca s-ar fi implicat Israelul sa reglementeze re
[CălinGeorgescu-CanalulOficial] Dumnezeu să vă binecuvinteze domnule Călin GEORGESCU;sânteți adevăratul președin


### Pasul 1 Alege două axe

Alege două axe pe care vrei să le codezi.
Poți folosi axe din curs:
- institutional
- legitimare
- epistemic
- geopolitic
- mobilizare
Sau poți propune axe proprii:
- media_distrust
- elite_blame
- religious_frame
- fear
- irony
- people_vs_elite
- anti_corruption
- national_identity
Condiție: fiecare axă trebuie să aibă valori clare.
Pentru acest exercițiu folosim o scală simplă:
0 = absent
1 = prezent


In [4]:
# modifica dupa preferinte

AXA_1 = "fear"
AXA_2 = "anti_corruption"

## Pasul 2 — Definește axele
Scrie mai jos, în propriile cuvinte, ce înseamnă fiecare axă.
Exemplu:
media_distrust = comentariul exprimă neîncredere în presă, jurnaliști, televiziuni sau media mainstream.
religious_frame = comentariul folosește limbaj religios pentru a interpreta politica.

In [5]:
AXA_1_DEFINITION = """
fear masoara daca textul incearca sa provoace frica, panica sau ingrijorare
despre o persoana, un grup sau o situatie.
0 = absent
1 = prezent
2 = dominant
"""
AXA_2_DEFINITION = """
anti_corruption masoara dacă textul exprima o atentie fata de coruptie sau abuzuri de putere.
0 = absent
1 = prezent
2 = dominant
"""

## Pasul 3 — Construiește mini-promptul
Promptul trebuie să conțină:
1. rolul modelului;
2. sarcina;
3. definițiile celor două axe;
4. regulile de codare;
5. formatul JSON.
Important:
- nu cere modelului să identifice direct „bula”;
- nu cere text liber;
- returnează doar JSON valid.

In [6]:
MINI_PROMPT = f"""
Ești un model de adnotare pentru comentarii politice.
SARCINĂ:
Adnotează comentariul folosind două axe:
1. {AXA_1}
2. {AXA_2}
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
{AXA_1} = 0 / 1 / 2
{AXA_2} = 0 / 1 / 2
DEFINIȚII:
{AXA_1_DEFINITION}
{AXA_2_DEFINITION}
REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
5. Pentru axe: 0 = absent, 1 = prezent, 2 = dominant.
6. Nu atribui direct o bulă discursivă.
7. Returnează doar JSON valid.
FORMAT OUTPUT:
{{
  "target": "",
  "stance": "",
  "tone": "",
  "{AXA_1}": 0,
  "{AXA_2}": 0
}}
"""
print(MINI_PROMPT)


Ești un model de adnotare pentru comentarii politice.
SARCINĂ:
Adnotează comentariul folosind două axe:
1. fear
2. anti_corruption
CÂMPURI:
target = ținta politică principală din comentariu
stance = poziția față de target: pro / anti / neutru / ambiguu / none
tone = modul dominant de formulare: acuzator / ironic / mobilizator / defensiv / afectiv / neutru
fear = 0 / 1 / 2
anti_corruption = 0 / 1 / 2
DEFINIȚII:

fear masoara daca textul incearca sa provoace frica, panica sau ingrijorare
despre o persoana, un grup sau o situatie.
0 = absent
1 = prezent
2 = dominant


anti_corruption masoara dacă textul exprima o atentie fata de coruptie sau abuzuri de putere.
0 = absent
1 = prezent
2 = dominant

REGULI:
1. Codează doar ce apare în comentariu, titlu sau canal.
2. Nu inventa informații externe.
3. Dacă nu există target politic, folosește target="none" și stance="none".
4. Dacă textul este ironic, codează sensul intenționat, nu sensul literal.
5. Pentru axe: 0 = absent, 1 = prezent, 2 = do

## Pasul 4 — Alege 5 comentarii de test
Folosim un eșantion mic. Nu adnotăm tot corpusul.
Schimbă `random_state` ca să primești alte comentarii.

In [11]:
TESTS = corpus.sample(5, random_state=55)
TESTS[["id", "source_channel", "video_title", "text"]].head()

,id,source_channel,video_title,text
104,yt_yEuctxNb4O0_UgzN5GlVQ60o-bYCC914AaABAg,NicusorDanRO,🟢 LIVE - Întâlnire la Palatul Cotroceni cu mag...,S-a transmis speranta in aceasta dezbatere. Es...
139,yt_Sj4fQKlMOro_UgyUQcT-gXyrgTWCVO14AaABAg,RecorderRomania,Lecție de curaj. Conferința care a zguduit jus...,Vă rog să aveți îndurare. Nu vreau să își piar...
374,yt_joXkZDqGZQU_Ugy5YoBCrvtEfT66whN4AaABAg,NicusorDanRO,🟢 Declarații de presă comune cu Președintele U...,M-a cam dezamagit președintele Dan în ultimul ...
30,yt_2lB4tTcjHzw_Ugz0V4O1fpH_HwAB2-F4AaABAg,euronewsro,Viktor Orban acuză UE și Ucraina că încearcă s...,Ii e frica lui Orban ca perioada lui de domnie...
167,yt_eC6XnF_Jy5k_Ugz-ZmJnbU7IA40x5Ch4AaABAg,NicusorDanRO,🟢 LIVE - Declarații de presă susținute după de...,Minte nicușoare că merită. Eu că am ales pe al...


## Pasul 5 — Rulează promptul pe cele 5 comentarii
Pentru fiecare comentariu:
1. trimitem canalul, titlul video și textul;
2. modelul returnează JSON;
3. citim rezultatul și verificăm dacă are sens.

In [12]:
USE_GEMINI = True
client_now = gemini_client if USE_GEMINI else deepseek_client
model_now = GEMINI_MODEL if USE_GEMINI else DEEPSEEK_MODEL
print("Using:", model_now)

Using: gemini-2.5-flash-lite


In [13]:
def llm(system, user, max_tokens=700):
    response = client_now.chat.completions.create(
        model=model_now,
        temperature=0,
        max_tokens=max_tokens,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user}
        ]
    )
    return response.choices[0].message.content

In [10]:
results = []
for _, row in TESTS.iterrows():
    USER = f"""
CANAL:
{row.get("source_channel", "")}
TITLU VIDEO:
{row.get("video_title", "")}
COMENTARIU:
<<< {row["text"]} >>>
"""
    raw = llm(MINI_PROMPT, USER, max_tokens=300)
    print("=" * 80)
    print("COMENTARIU:")
    print(row["text"])
    print()
    print("OUTPUT MODEL:")
    print(raw)
    results.append({
        "id": row["id"],
        "text": row["text"],
        "model_output": raw
    })

COMENTARIU:
S-a transmis speranta in aceasta dezbatere. Este foarte important in urma concluziilor finale sa se poata imbunatatii si insanatosi justitia.

OUTPUT MODEL:
```json
{
  "target": "justitia",
  "stance": "pro",
  "tone": "afectiv",
  "fear": 0,
  "anti_corruption": 1
}
```
COMENTARIU:
Vă rog să aveți îndurare. Nu vreau să își piardă nimeni serviciul. SE și obligația Justiției de a prezenta adevărul post-mortem prin hotărâri judecătorești.

OUTPUT MODEL:
```json
{
  "target": "justiția din România",
  "stance": "ambiguu",
  "tone": "afectiv",
  "fear": 1,
  "anti_corruption": 1
}
```
COMENTARIU:
M-a cam dezamagit președintele Dan în ultimul timp în special în ceea ce privește procurorii, părerile despre CSM și atitudinea aroganță de la ultima manifestație, însă unde este de laudat îl voi lăuda. Discursul a fost foarte bun, întâlnirea productivă și am simțit un sentiment de mândrie că acesta este președintele României.

OUTPUT MODEL:
```json
{
  "target": "Klaus Iohannis",
  "

Am ales axele fear si anti_corruption deoarece tema fricii este mai sensibila si mai putin discutata direct in comentarii, iar anti_corruption apare mult mai des in discursul politic si este mai usor de observat. 
Modelul a returnat JSON corect si a respectat formatul cerut. 
Cea mai mare problema a fost faptul ca unele rezultate au fost usor ambigue, mai ales la comentariile ironice, dar in general axele au putut fi identificate corect. 
Ca imbunatatire, as face regulile pentru scorurile 0, 1 si 2 mai detaliate si as adauga mai multe exemple pentru fiecare situatie.